<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 12: Maske Tespiti

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 12 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta12/hafta12_maske_tespiti.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta12/hafta12_maske_tespiti.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>

</div>

# Hafta 12 - Yüz Maskesi Tespiti

Bu defterde sentetik veri seti oluşturarak yüz maskesi tespiti yapacağız.

## İçerik
1. Sentetik Veri Seti Oluşturma
2. Veri Artırma (Data Augmentation)
3. CNN ile Maske Tespiti Modeli
4. Transfer Öğrenme ile Maske Tespiti
5. Değerlendirme ve Karışıklık Matrisi

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow sürümü: {tf.__version__}")

## 1. Gerçek Face Mask Detection Verisini Yükleme

**Face Mask Detection Dataset** — 12.000 gerçek yüz görüntüsü (maskeli ve maskesiz). Eğitim ve test setleri olarak ayrılmış.

**Kaynak:** [Kaggle - Face Mask 12K Images](https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset)

### Yükleme Yöntemleri

**Yöntem 1:** Kaggle API ile (önerilir)
```python
!pip install kaggle
!kaggle datasets download -d ashishjangra27/face-mask-12k-images-dataset
!unzip face-mask-12k-images-dataset.zip
```

**Yöntem 2:** opendatasets ile
```python
!pip install opendatasets
import opendatasets as od
od.download("https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset")
```

**Yöntem 3:** Google Colab'a manuel yükleme

In [ ]:
# Face Mask Detection Dataset yükleme
!pip install -q opendatasets

import opendatasets as od
import os

# Kaggle'dan indir (API key gerekir — ilk seferde sorar)
dataset_url = "https://www.kaggle.com/datasets/ashishjangra27/face-mask-12k-images-dataset"

# Eğer zaten indirildiyse tekrar indirme
dataset_dir = "face-mask-12k-images-dataset"
if not os.path.exists(dataset_dir):
    od.download(dataset_url)
    print("Veri seti indirildi!")
else:
    print("Veri seti zaten mevcut.")

# Klasör yapısını kontrol et
train_dir = os.path.join(dataset_dir, "Face Mask Dataset", "Train")
test_dir = os.path.join(dataset_dir, "Face Mask Dataset", "Test")

if os.path.exists(train_dir):
    for sinif in os.listdir(train_dir):
        sinif_yolu = os.path.join(train_dir, sinif)
        if os.path.isdir(sinif_yolu):
            print(f"Eğitim - {sinif}: {len(os.listdir(sinif_yolu))} görüntü")
    for sinif in os.listdir(test_dir):
        sinif_yolu = os.path.join(test_dir, sinif)
        if os.path.isdir(sinif_yolu):
            print(f"Test - {sinif}: {len(os.listdir(sinif_yolu))} görüntü")

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Veri setini TensorFlow ile yükle
GORUNTU_BOYUTU = 128
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(GORUNTU_BOYUTU, GORUNTU_BOYUTU),
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
    seed=42
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(GORUNTU_BOYUTU, GORUNTU_BOYUTU),
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

sinif_isimleri = train_ds.class_names
print(f"Sınıflar: {sinif_isimleri}")

# Normalize et (0-1 arası)
normalization = tf.keras.layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization(x), y))
test_ds = test_ds.map(lambda x, y: (normalization(x), y))

# Performans için prefetch
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

# Örnek görüntüleri göster
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(min(12, len(images))):
        ax = plt.subplot(3, 4, i + 1)
        plt.imshow(images[i].numpy())
        sinif = sinif_isimleri[int(labels[i])]
        plt.title(f"{'Maskeli' if sinif == 'WithMask' else 'Maskesiz'}", fontsize=11)
        plt.axis('off')
plt.suptitle("Eğitim Setinden Örnek Görüntüler", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Veri Artırma

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `tensorflow` | Derin öğrenme modelleri oluşturma ve eğitme |


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

veri_artirma = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    zoom_range=0.1
)

# Artırılmış örnekler göster
ornek = X_train[0:1]
plt.figure(figsize=(15, 2))

plt.subplot(1, 8, 1)
plt.imshow(ornek[0])
plt.title('Orijinal', fontsize=9)
plt.axis('off')

artirma_iter = veri_artirma.flow(ornek, batch_size=1)
for i in range(7):
    plt.subplot(1, 8, i + 2)
    plt.imshow(artirma_iter.next()[0])
    plt.title(f'Artırılmış {i+1}', fontsize=9)
    plt.axis('off')

plt.suptitle('Veri Artırma Örnekleri', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. CNN ile Maske Tespiti Modeli

### Sinir Ağı Modeli Oluşturma

Keras Sequential API ile katman katman sinir ağı modeli inşa ediyoruz. Her katmanın kendine özgü bir görevi vardır (özellik çıkarma, boyut düşürme, sınıflandırma).

In [ ]:
# CNN modeli oluştur
cnn_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(GORUNTU_BOYUTU, GORUNTU_BOYUTU, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Modeli eğit
gecmis_cnn = cnn_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

## 4. Transfer Öğrenme ile Maske Tespiti (MobileNetV2)

In [ ]:
# Görüntüleri yeniden boyutlandır (MobileNetV2 için en az 96x96)
X_train_resized = tf.image.resize(X_train, (96, 96)).numpy()
X_test_resized = tf.image.resize(X_test, (96, 96)).numpy()

# MobileNetV2 taban modeli
taban_model = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(96, 96, 3)
)
taban_model.trainable = False

# Transfer öğrenme modeli
transfer_model = tf.keras.Sequential([
    taban_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Transfer Öğrenme modeli hazır.")
print(f"Eğitilebilir parametre: {sum(tf.keras.backend.count_params(w) for w in transfer_model.trainable_weights):,}")

### Model Eğitimi

Aşağıdaki kodda modeli eğitim verisi üzerinde eğitiyoruz (`.fit()`). Eğitim sonrası test verisi üzerinde tahmin yapıp (`.predict()`) başarı metriklerini hesaplıyoruz.

In [ ]:
# Transfer öğrenme modelini eğit
gecmis_transfer = transfer_model.fit(
    X_train_resized, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test_resized, y_test),
    verbose=1
)

## 5. Değerlendirme ve Karışıklık Matrisi

### Her iki modeli değerlendir

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Her iki modeli değerlendir
cnn_kayip, cnn_dogruluk = cnn_model.evaluate(X_test, y_test, verbose=0)
transfer_kayip, transfer_dogruluk = transfer_model.evaluate(X_test_resized, y_test, verbose=0)

print("=" * 50)
print("MASKE TESPİTİ MODEL KARŞILAŞTIRMASI")
print("=" * 50)
print(f"{'Model':<30} {'Doğruluk':>12}")
print("-" * 50)
print(f"{'CNN (sıfırdan)':<30} {cnn_dogruluk*100:>11.2f}%")
print(f"{'Transfer Öğrenme (MobileNetV2)':<30} {transfer_dogruluk*100:>11.2f}%")
print("=" * 50)

### Dağılım Görselleştirmesi

Aşağıdaki grafikte verinin dağılımını histogram ile inceliyoruz. Dağılımın şekli (normal, çarpık, bimodal) hangi istatistiksel yöntemlerin uygulanabileceğini belirler.

In [ ]:
# Eğitim grafikleri
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(gecmis_cnn.history['accuracy'], label='CNN Eğitim', linewidth=2)
axes[0].plot(gecmis_cnn.history['val_accuracy'], label='CNN Doğrulama', linewidth=2, linestyle='--')
axes[0].plot(gecmis_transfer.history['accuracy'], label='Transfer Eğitim', linewidth=2)
axes[0].plot(gecmis_transfer.history['val_accuracy'], label='Transfer Doğrulama', linewidth=2, linestyle='--')
axes[0].set_title('Doğruluk Karşılaştırması', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Doğruluk')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].plot(gecmis_cnn.history['loss'], label='CNN Eğitim', linewidth=2)
axes[1].plot(gecmis_cnn.history['val_loss'], label='CNN Doğrulama', linewidth=2, linestyle='--')
axes[1].plot(gecmis_transfer.history['loss'], label='Transfer Eğitim', linewidth=2)
axes[1].plot(gecmis_transfer.history['val_loss'], label='Transfer Doğrulama', linewidth=2, linestyle='--')
axes[1].set_title('Kayıp Karşılaştırması', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Kayıp')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Korelasyon Analizi ve Isı Haritası

Özellikler arasındaki ilişkiyi korelasyon matrisi ve ısı haritası ile görselleştiriyoruz. Yüksek korelasyonlu özellikler modelin en çok yararlanacağı özelliklerdir.

In [ ]:
# Karışıklık Matrisi (CNN modeli için)
cnn_tahminler = (cnn_model.predict(X_test) > 0.5).astype(int).flatten()
cm = confusion_matrix(y_test.astype(int), cnn_tahminler)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Maskesiz', 'Maskeli'],
            yticklabels=['Maskesiz', 'Maskeli'])
plt.title('CNN Modeli - Karışıklık Matrisi', fontsize=14, fontweight='bold')
plt.xlabel('Tahmin Edilen')
plt.ylabel('Gerçek')
plt.tight_layout()
plt.show()

print("\nSınıflandırma Raporu (CNN):")
print(classification_report(y_test.astype(int), cnn_tahminler,
                            target_names=['Maskesiz', 'Maskeli']))

### Çoklu Grafik Paneli

Birden fazla grafiği yan yana veya alt alta çizdirerek karşılaştırmalı analiz yapıyoruz.

In [ ]:
# Örnek tahminleri görselleştir
etiket_isimleri = ['Maskesiz', 'Maskeli']
test_tahminler = cnn_model.predict(X_test[:20])

plt.figure(figsize=(15, 6))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(X_test[i])
    
    tahmin_idx = int(test_tahminler[i] > 0.5)
    gercek_idx = int(y_test[i])
    guvence = test_tahminler[i][0] if tahmin_idx == 1 else 1 - test_tahminler[i][0]
    
    renk = 'green' if tahmin_idx == gercek_idx else 'red'
    plt.title(f"{etiket_isimleri[tahmin_idx]}\n%{guvence*100:.0f}", fontsize=8, color=renk)
    plt.axis('off')

plt.suptitle('Maske Tespiti Tahminleri', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Özet

Bu defterde öğrendiklerimiz:

1. **Sentetik Veri Oluşturma**: Gerçek veri yokken sentetik veriyle çalışma
2. **Veri Artırma**: Eğitim verisini çeşitlendirme teknikleri
3. **CNN Modeli**: Sıfırdan evrişimli sinir ağı eğitimi
4. **Transfer Öğrenme**: MobileNetV2 ile maske tespiti
5. **Değerlendirme**: Karışıklık matrisi ve sınıflandırma raporu

### Gerçek Dünya Uygulaması
- Gerçek projede kaggle.com'dan yüz maskesi veri setleri kullanılabilir
- Kameradan gerçek zamanlı maske tespiti OpenCV ile yapılabilir
- Daha yüksek çözünürlüklü görüntülerle model performansı artırılabilir

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

© 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>